## Exercise 5: Geospatial wrangling and making maps

Skills: 
* More geospatial practice building on earlier skills
* Make a map with `folium` or `ipyleaflet` using `shared_utils.map_utils`

References: 
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html

In [1]:
import geopandas as gpd
import intake
import pandas as pd

from calitp.tables import tbl
from calitp import query_sql
from siuba import *

from shapely.geometry import Polygon, LineString, Point
from shapely import wkt
from shapely.geometry import Point, Polygon
from shapely.wkt import loads

import shared_utils
import altair as alt
from shared_utils import altair_utils 

import branca
import folium
# Hint: if this doesn't import: refer to docs for correctly import
# cd into _shared_utils folder, run the make setup_env command

pd.options.display.float_format = "{:.2f}".format

/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.9.1-CAPI-1.14.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(


## Research Question: What's the average number of trips per stop by operators in southern California? Show visualizations at the operator and county-level.
<br>**Geographic scope:** southern California counties
<br>**Deliverables:** chart(s) and map(s) showing metrics comparing across counties and also across operators. Make these visualizations using function(s).

### Prep data

* Use the same query, but grab a different set of operators. These are in southern California, so the map should zoom in counties ranging from LA to SD.
* To find what ITP IDs are what operators:
[agencies.yml](https://github.com/cal-itp/data-infra/blob/main/airflow/data/agencies.yml)
* *Hint*: for some counties, there are multiple operators. Make sure the average trips per stop by counties is the weighted average.
* Use the same [shapefile for CA counties](https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-county-boundaries/explore?location=37.246136%2C-119.002032%2C6.12) as in Exercise 4.
* Join the data and only keep counties that have bus stops.


In [2]:
#choosing a different set of ITP IDS
ITP_ID = [182, 183, 278, 154, 235, 232]

In [3]:
stops = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
    >> distinct()
)

/opt/conda/lib/python3.9/site-packages/siuba/sql/utils.py:52: SAWarning: Dialect bigquery:bigquery will not make use of SQL compilation caching as it does not set the 'supports_statement_cache' attribute to ``True``.  This can have significant performance implications including some performance degradations in comparison to prior SQLAlchemy versions.  Dialect maintainers should seek to set this attribute to True after appropriate development and testing for SQLAlchemy 1.4 caching support.   Alternatively, this attribute may be set to False which will disable this warning. (Background on this error at: https://sqlalche.me/e/14/cprf)


In [4]:
#creating point geometry from longtitude &  latitude 
stops = shared_utils.geography_utils.create_point_geometry(stops, 'stop_lon', 'stop_lat')

In [5]:
stops.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry
0,154,19113,33.54,-117.78,Laguna Beach Bus Depot,POINT (-117.78271 33.54487)
1,154,19114,33.54,-117.78,Legion Hall,POINT (-117.78006 33.54069)


In [6]:
#checking CRS
stops.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [7]:
#bringing in CA dataframe
geojson = gpd.read_file('https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson').to_crs(epsg=4326)

In [8]:
#Joining the 2 dataframes, only keeping stops that are in a county
stops_joined = gpd.sjoin(stops, geojson, how='inner')

### Bring in a new table from BigQuery

* In `transitstacks`, there's a table called `ntd_stats`. 
* Decide what columns to keep.
* Merge `ntd_stats` with the `stops` table to create 1 df.

In [9]:
ntd_stats = (tbl.transitstacks.ntd_stats()
             >> select(_.transit_provider, _.itp_id, _.upt_total_2019)
             >> collect()
             >> filter(_.itp_id.isin(ITP_ID))
            )

In [10]:
#getting rid of commas & assigning to int to sum up later. 
ntd_stats = ntd_stats.assign(
    upt_total_2019 = ntd_stats.upt_total_2019.replace(',','', regex=True)).astype({"upt_total_2019": int}) 

### Merging the stops with ntd_stats 

In [11]:
df2 = stops_joined.merge(ntd_stats, how = 'left', on ='itp_id')

In [12]:
f'out of the {len(df2)} rows, only {df2.stop_id.nunique()} are unique'

'out of the 44596 rows, only 22356 are unique'

In [13]:
#new dataframe with only unique stop ids & trips per stop
df3 = df2.drop_duplicates(subset=['stop_id','itp_id'])

### Aggregate 
<b> Instructions </b> 
* Write a function to aggregate to the operator level or county level, add new columns for desired metrics.
* Merge in CA shapefile to get a gdf.
* Add another `geometry` column, called `centroid`, and grab the county's centroid.
* Refer to [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.set_geometry.html) to see how to pick which column to use as the `geometry` for the gdf, since technically, a gdf can handle multiple geometry columns.



In [15]:
df3.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,transit_provider,upt_total_2019
0,154,19113,33.54,-117.78,Laguna Beach Bus Depot,POINT (-117.78271 33.54487),29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,Laguna Beach Municipal Transit,820829
1,154,19114,33.54,-117.78,Legion Hall,POINT (-117.78006 33.54069),29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,Laguna Beach Municipal Transit,820829


In [16]:
#County Subsets
county_stops = df3[['geometry','stop_id','COUNTY_NAME']]
county_passengers = df3[['geometry','upt_total_2019','COUNTY_NAME']]

In [17]:
#transit providers
transit_stops = df3[['geometry','stop_id','transit_provider']]
transit_passengers = df3[['geometry','upt_total_2019','transit_provider']]

In [18]:
#my function
def aggregate2 (df1, df2, group_by_col):
    df1 = df1.dissolve(by = group_by_col, aggfunc= 'nunique').rename(columns = {'stop_id':'total_stops'})
    df2 = df2.dissolve(by = group_by_col, aggfunc= 'sum').rename(columns = {'upt_total_2019':'total_trips'})
    df3 = df1.merge(df2, how = 'inner', on = [group_by_col, 'geometry'])
    #find sum up total trips 
    df3['total_trips_sum'] = df3['total_trips'].sum() 
    #multiply total stops by total trips within a transit operator / county  
    df3['frequency'] = df3['total_stops']*df3['total_trips']
    #divide
    df3['weighted_avg'] = df3['frequency']/df3['total_trips_sum'] 
    df3 = df3.reset_index()
    return df3

In [19]:
#Tiffany's function
def aggregate3(df, group_by_col):
    df1 = df.groupby(group_by_col).agg({"stop_id": "nunique", 
                                           "upt_total_2019": "sum"}).reset_index()
    df1 = df1.rename(columns = {'stop_id':'total_stops', 'upt_total_2019':'total_trips'})
    df1['total_trips_sum'] = df1['total_trips'].sum() 
    df1['frequency'] = df1['total_stops']*df1['total_trips']
    df1['weighted_avg'] = df1['frequency']/df1['total_trips_sum'] 
    df2 = df[['geometry',group_by_col]].drop_duplicates(subset = [group_by_col]) 
    df3 = pd.merge(df1, 
                   df2, 
                   on = group_by_col, 
                   how = "left", 
                   validate = "1:1",
                  )
    
    return df3

In [20]:
#my function
county_agg = aggregate2(county_stops,county_passengers, 'COUNTY_NAME')
county_agg

,COUNTY_NAME,geometry,total_stops,total_trips,total_trips_sum,frequency,weighted_avg
0,Los Angeles,"MULTIPOINT (-118.84339 34.03152, -118.82790 34...",16903,5316430995587,5982206873784,89863633118407061,15021.82
1,Orange,"MULTIPOINT (-118.10607 33.74889, -118.10577 33...",5581,239217387671,5982206873784,1335072240591851,223.17
2,Riverside,"MULTIPOINT (-117.55461 33.99433, -117.46784 33...",5,84197774,5982206873784,420988870,0.00
3,San Bernardino,"MULTIPOINT (-117.73260 34.00136, -117.73217 33...",2331,25322888430,5982206873784,59027652930330,9.87
4,San Diego,"MULTIPOINT (-117.27792 32.83915, -117.27780 32...",4569,389998394655,5982206873784,1781902665178695,297.87
5,Ventura,"MULTIPOINT (-118.88283 34.18006, -118.88280 34...",55,11153009667,5982206873784,613415531685,0.10


In [21]:
#Tiffany's function
county_agg_v2 = aggregate3(df3, 'COUNTY_NAME')
county_agg_v2

,COUNTY_NAME,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,geometry
0,Los Angeles,16903,5316430995587,5982206873784,89863633118407061,15021.82,POINT (-118.11311 33.97325)
1,Orange,5581,239217387671,5982206873784,1335072240591851,223.17,POINT (-117.78271 33.54487)
2,Riverside,5,84197774,5982206873784,420988870,0.00,POINT (-117.37136 33.97539)
3,San Bernardino,2331,25322888430,5982206873784,59027652930330,9.87,POINT (-117.35178 34.07250)
4,San Diego,4569,389998394655,5982206873784,1781902665178695,297.87,POINT (-117.24043 32.67446)
5,Ventura,55,11153009667,5982206873784,613415531685,0.10,POINT (-118.84884 34.17603)


In [22]:
#Tiffany's function
transit_agg = aggregate3(df3, 'transit_provider')
transit_agg

,transit_provider,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,geometry
0,Laguna Beach Municipal Transit,94,77157926,5982206873784,7252845044,0.00,POINT (-117.78271 33.54487)
1,Los Angeles Department of Transportation,3033,58514689341,5982206873784,177475052771253,29.67,POINT (-118.36986 33.89505)
2,Los Angeles Metro,13906,5280360190626,5982206873784,73428688810845156,12274.52,POINT (-117.91591 33.81036)
3,OmniTrans,2351,25540159030,5982206873784,60044913879530,10.04,POINT (-117.75124 34.05917)
4,Orange County Transportation Authority,5589,227716282206,5982206873784,1272706301249334,212.75,POINT (-117.90981 33.82344)
5,San Diego Metropolitan Transit System,4569,389998394655,5982206873784,1781902665178695,297.87,POINT (-117.24043 32.67446)


#### Centroids

In [23]:
#merging for geojson again, dropping and renaming cols - using my df first 
county_agg2 = pd.merge(county_agg, geojson,  on=['COUNTY_NAME']).drop(columns = ['geometry_x']).rename(columns = {'COUNTY_NAME':'County', 'geometry_y':'geometry'}) 
#filter out for islands
county_agg2 = county_agg2.loc[county_agg2['ISLAND'] != 'Channel Islands'] 
#find center
county_agg2['centroid'] = county_agg2.geometry.centroid
#preview 
county_agg2[['County','centroid','weighted_avg']]

/tmp/ipykernel_862/1062378201.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



,County,centroid,weighted_avg
0,Los Angeles,POINT (-118.21689 34.36117),15021.82
3,Orange,POINT (-117.76121 33.70309),223.17
4,Riverside,POINT (-115.99382 33.74374),0.00
5,San Bernardino,POINT (-116.17852 34.84146),9.87
6,San Diego,POINT (-116.73509 33.03422),297.87
7,Ventura,POINT (-119.07833 34.47124),0.10


In [59]:
#using Tiffany's function
county_agg3 = pd.merge(county_agg_v2, geojson,  on=['COUNTY_NAME']).drop(columns = ['geometry_x']).rename(columns = {'COUNTY_NAME':'County', 'geometry_y':'geometry'}) 
#filter out for islands
county_agg3 = county_agg2.loc[county_agg2['ISLAND'] != 'Channel Islands'] 
#find center
county_agg3['centroid'] = county_agg2.geometry.centroid
#preview 
county_agg3[['County','centroid','weighted_avg']]

/tmp/ipykernel_862/11312977.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



,County,centroid,weighted_avg
0,Los Angeles,POINT (-118.21689 34.36117),15021.82
3,Orange,POINT (-117.76121 33.70309),223.17
4,Riverside,POINT (-115.99382 33.74374),0.00
5,San Bernardino,POINT (-116.17852 34.84146),9.87
6,San Diego,POINT (-116.73509 33.03422),297.87
7,Ventura,POINT (-119.07833 34.47124),0.10


## Visualizations
* Make one chart for comparing trips per stop by operators, and another chart for comparing it by counties. Use a function to do this.
* Make 1 map for comparing trips per stop by counties. Use `shared_utils.map_utils` to do this.
* Visualizations should follow the Cal-ITP style guide.
* More on `folium` and `ipyleaflet`: https://github.com/jorisvandenbossche/geopandas-tutorial/blob/master/05-more-on-visualization.ipynb

### Scatter Chart Function

In [24]:
#scatter chart
def scatter_chart(df, x_col, y_col):
    y_col_stripped = y_col.replace('_',' ')
    x_col_stripped = x_col.replace('_',' ')
    chart_title = f"{y_col_stripped} by {x_col_stripped}" 
    chart = (alt.Chart(df).mark_circle(size=500).encode(
                 x=alt.X(x_col, title=x_col_stripped),
                 y=alt.Y(y_col, title=y_col_stripped),   
                 color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.CALITP_CATEGORY_BOLD_COLORS)),
                 tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)]).interactive().properties(width=600,height=250,
                 title = chart_title)
            )
    #return chart
    display(chart)

In [25]:
#function to make all my charts 
def charts(df, x_axis_col):
    for i in ['total_stops', 'total_trips','weighted_avg']:
        df_i = df[[x_axis_col, i]].drop_duplicates()
        scatter_chart_i = scatter_chart(df_i, x_axis_col, i)
        display (scatter_chart_i)
    return scatter_chart_i

In [26]:
charts(transit_agg, 'transit_provider')

alt.Chart(...)

None

alt.Chart(...)

None

alt.Chart(...)

None

In [27]:
charts(county_agg2, 'County')

alt.Chart(...)

None

alt.Chart(...)

None

alt.Chart(...)

None

### Map

In [50]:
choropleth_dict = ({
            "layer_name": 'my layer',
            "MIN_VALUE": int(0),
            "MAX_VALUE": int(100),
            "plot_col_name": 'County_Name',
            "fig_width": '100%',
            "fig_height": '100%',
            "fig_min_width_px": '600px',
            "fig_min_height_px": '600px'})

In [60]:
#removing centroid because the map throws an error.
county_agg2 = county_agg2.drop(columns = ['centroid']).rename(columns = {'geometry_y':'geometry'})

In [61]:
county_agg3 = county_agg3.drop(columns = ['centroid']).rename(columns = {'geometry_y':'geometry'})

In [52]:
colorscale = branca.colormap.StepColormap(
                colors=["#F6BF16", "#E16B26",  "#00896B"], 
                index=[0, 116, 15_000], #looking at the percentiles to play with color scale
                vmin=0, vmax=15_000,
)
colorscale

In [53]:
county_agg2.weighted_avg.describe()

count       6.00
mean     2592.14
std      6090.63
min         0.00
25%         2.54
50%       116.52
75%       279.19
max     15021.82
Name: weighted_avg, dtype: float64

### HELP, how do I get region centroids to show complete list of counties in CA?

In [54]:
REGION_CENTROIDS = shared_utils.map_utils.REGION_CENTROIDS

In [55]:
REGION_CENTROIDS

{'Alameda': [[37.84, -122.27], 12],
 'Los Angeles': [[34.0, -118.18], 11],
 'CA': [[35.8, -119.4], 6]}

In [62]:
shared_utils.map_utils.make_ipyleaflet_choropleth_map(county_agg3, 
                                                     plot_col = 'weighted_avg',
                                                     geometry_col = 'County', 
                                                     choropleth_dict = choropleth_dict, 
                                                     colorscale = colorscale,
                                                     zoom=7, 
                                                     centroid = REGION_CENTROIDS['Los Angeles'][0])

Map(center=[34.0, -118.18], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…